In [ ]:
# ==========================================
# STEP 1. 라이브러리
# ==========================================

import os
import zipfile
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

print("라이브러리 로드 완료")

In [ ]:
# ==========================================
# STEP 2. ZIP 압축 해제
# ==========================================

zip_path = "/content/fraud_full_features.zip"
extract_path = "/content/fraud_full_features"

os.makedirs(
    extract_path,
    exist_ok=True
)

with zipfile.ZipFile(
    zip_path,
    "r"
) as zip_ref:

    zip_ref.extractall(
        extract_path
    )

print("압축 해제 완료")

print("\n압축 해제된 파일:")
print(
    os.listdir(
        extract_path
    )
)

In [ ]:
# ==========================================
# STEP 3. 데이터 로드
# ==========================================

df = pd.read_csv(
    "/content/fraud_full_features/fraud_full_features.csv"
)

print("데이터 로드 완료")

print(
    "전체 데이터:",
    df.shape
)

print(
    "전체 컬럼:",
    len(df.columns)
)

In [ ]:
# ==========================================
# STEP 4. 시간순 정렬
# ==========================================

df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"]
)

df = (
    df
    .sort_values(
        "trans_date_trans_time"
    )
    .reset_index(drop=True)
)

print("전체 기간")
print(
    df["trans_date_trans_time"].min(),
    "~",
    df["trans_date_trans_time"].max()
)

print(
    "\n전체 거래:",
    len(df)
)

print(
    "전체 이상거래:",
    int(df["is_fraud"].sum())
)

print(
    "전체 이상거래율:",
    f"{df['is_fraud'].mean() * 100:.4f}%"
)

In [ ]:
# ==========================================
# STEP 5. Feature Set 정의
# ==========================================

set1 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min"
]


set2 = set1 + [
    "merchant_change_count"
]


set3 = set1 + [
    "high_speed"
]


set4 = [
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "category",
    "amt",
    "trans_hour",
    "age"
]


set5 = [
    "category",
    "amt",
    "is_online",
    "recent_24h_high_amt_count",
    "category_recent_fraud_rate",
    "speed_2",
    "customer_mean_amt",
    "customer_std_amt",
    "amt_ratio_to_mean",
    "amt_zscore_card",
    "customer_transaction_count",
    "trans_hour",
    "age",
    "rolling_sum_amt_1h",
    "prior_normal_median_amt",
    "amt_to_prior_median_ratio",
    "risk_time_22_04",
    "interact_repeat_category",
    "has_prior_normal_transaction"
]


feature_sets = {
    "Set 1": set1,
    "Set 2": set2,
    "Set 3": set3,
    "Set 4": set4,
    "Set 5": set5
}


for name, features in feature_sets.items():

    print(
        name,
        ":",
        len(features),
        "개"
    )

In [ ]:
# ==========================================
# STEP 6. 변수 존재 확인
# ==========================================

for set_name, features in feature_sets.items():

    missing = [
        x
        for x in features
        if x not in df.columns
    ]

    if not missing:

        print(
            f"✅ {set_name}: "
            f"모든 변수 존재"
        )

    else:

        print(
            f"❌ {set_name}: "
            f"{missing}"
        )

In [ ]:
# ==========================================
# STEP 7. 결측치 확인
# ==========================================

all_features = sorted(
    set(
        feature
        for features in feature_sets.values()
        for feature in features
    )
)

missing_count = (
    df[all_features]
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

display(
    missing_count[
        missing_count > 0
    ]
)

In [ ]:
# ==========================================
# STEP 8. 공통 실험 설정
# ==========================================

RANDOM_STATE = 42
THRESHOLD = 0.90

# 팀 공통 불균형 처리 가중치
normal = (y_train == 0).sum()
fraud = (y_train == 1).sum()

scale_pos_weight = normal / fraud

class_weight = {
    0: 1.0,
    1: scale_pos_weight
}

# 학습 중단 조건
MAX_ITER = 100


print("Threshold           :", THRESHOLD)
print("Positive Class Weight:", POS_WEIGHT)
print("Max Iteration       :", MAX_ITER)
print("Random State        :", RANDOM_STATE)

In [ ]:
# ==========================================
# STEP 9. Logistic Pipeline
# ==========================================

CATEGORICAL_COLUMNS = [
    "category"
]


def make_logistic_pipeline(
    features,
    C=1.0,
    penalty="l2",
    solver="liblinear"
):

    categorical_features = [
        col
        for col in features
        if col in CATEGORICAL_COLUMNS
    ]

    numeric_features = [
        col
        for col in features
        if col not in CATEGORICAL_COLUMNS
    ]

    # ----------------------------
    # 수치형
    # ----------------------------

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])

    # ----------------------------
    # 범주형
    # ----------------------------

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ])

    logistic = LogisticRegression(

        C=C,

        penalty=penalty,

        solver=solver,

        class_weight={
            0: 1,
            1: POS_WEIGHT
        },

        max_iter=MAX_ITER,

        random_state=RANDOM_STATE
    )

    model = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            logistic
        )
    ])

    return model

In [ ]:
# ==========================================
# STEP 10. 평가 함수
# ==========================================

def evaluate_model(
    y_true,
    probability,
    threshold=THRESHOLD
):

    prediction = (
        probability >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1]
    ).ravel()

    return {

        "PR-AUC":
            average_precision_score(
                y_true,
                probability
            ),

        "ROC-AUC":
            roc_auc_score(
                y_true,
                probability
            ),

        "Accuracy":
            accuracy_score(
                y_true,
                prediction
            ),

        "Precision":
            precision_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "F1":
            f1_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "FP": fp,
        "FN": fn,
        "TP": tp,
        "TN": tn
    }

In [ ]:
# ==========================================
# STEP 11. 시간순 80:20
# ==========================================

split_index = int(
    len(df) * 0.8
)

train_df_80 = (
    df
    .iloc[:split_index]
    .copy()
)

val_df_20 = (
    df
    .iloc[split_index:]
    .copy()
)


print("===== Train 80% =====")

print(
    "거래:",
    len(train_df_80)
)

print(
    "Fraud:",
    int(
        train_df_80[
            "is_fraud"
        ].sum()
    )
)

print(
    "Fraud Rate:",
    f"{train_df_80['is_fraud'].mean()*100:.4f}%"
)


print("\n===== Validation 20% =====")

print(
    "거래:",
    len(val_df_20)
)

print(
    "Fraud:",
    int(
        val_df_20[
            "is_fraud"
        ].sum()
    )
)

print(
    "Fraud Rate:",
    f"{val_df_20['is_fraud'].mean()*100:.4f}%"
)

In [ ]:
# ==========================================
# STEP 12. 80:20 Feature Set 비교
# ==========================================

results_80 = []


for set_name, features in feature_sets.items():

    print("\n" + "=" * 70)
    print(set_name)
    print("=" * 70)

    X_train = train_df_80[
        features
    ]

    y_train = train_df_80[
        "is_fraud"
    ]

    X_val = val_df_20[
        features
    ]

    y_val = val_df_20[
        "is_fraud"
    ]


    model = make_logistic_pipeline(
        features=features,
        C=1.0,
        penalty="l2",
        solver="liblinear"
    )


    model.fit(
        X_train,
        y_train
    )


    val_prob = model.predict_proba(
        X_val
    )[:, 1]


    metrics = evaluate_model(
        y_val,
        val_prob
    )


    results_80.append({

        "Feature Set":
            set_name,

        "N Features":
            len(features),

        **metrics
    })


results_80_df = pd.DataFrame(
    results_80
)


results_80_df = (
    results_80_df
    .sort_values(
        "PR-AUC",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    results_80_df
)

In [ ]:
# ==========================================
# STEP 13. 시간순 데이터 행 기준 6등분
# ==========================================

segments = np.array_split(
    df,
    6
)

segment_summary = []


for i, segment in enumerate(
    segments,
    start=1
):

    segment_summary.append({

        "Segment":
            i,

        "Transactions":
            len(segment),

        "Fraud":
            int(
                segment[
                    "is_fraud"
                ].sum()
            ),

        "Fraud Rate (%)":
            segment[
                "is_fraud"
            ].mean() * 100,

        "Start":
            segment[
                "trans_date_trans_time"
            ].min(),

        "End":
            segment[
                "trans_date_trans_time"
            ].max()
    })


segment_summary_df = pd.DataFrame(
    segment_summary
)


display(
    segment_summary_df
)

In [ ]:
# ==========================================
# STEP 14. 3-Fold Expanding Window
# ==========================================

folds = {

    "Fold 1": {
        "train_segments": [0, 1, 2],
        "val_segment": 3
    },

    "Fold 2": {
        "train_segments": [0, 1, 2, 3],
        "val_segment": 4
    },

    "Fold 3": {
        "train_segments": [0, 1, 2, 3, 4],
        "val_segment": 5
    }
}


def get_fold_data(
    segments,
    fold_info
):

    fold_train = pd.concat(
        [
            segments[i]
            for i in fold_info[
                "train_segments"
            ]
        ],
        axis=0
    ).copy()

    fold_val = (
        segments[
            fold_info[
                "val_segment"
            ]
        ]
        .copy()
    )

    return (
        fold_train,
        fold_val
    )

In [ ]:
# ==========================================
# STEP 15. Fold 확인
# ==========================================

fold_summary = []


for fold_name, fold_info in folds.items():

    fold_train, fold_val = (
        get_fold_data(
            segments,
            fold_info
        )
    )


    fold_summary.append({

        "Fold":
            fold_name,

        "Train N":
            len(fold_train),

        "Validation N":
            len(fold_val),

        "Train Fraud":
            int(
                fold_train[
                    "is_fraud"
                ].sum()
            ),

        "Validation Fraud":
            int(
                fold_val[
                    "is_fraud"
                ].sum()
            ),

        "Validation Fraud Rate (%)":
            fold_val[
                "is_fraud"
            ].mean() * 100,

        "Train End":
            fold_train[
                "trans_date_trans_time"
            ].max(),

        "Validation Start":
            fold_val[
                "trans_date_trans_time"
            ].min()
    })


fold_summary_df = pd.DataFrame(
    fold_summary
)


display(
    fold_summary_df
)

In [ ]:
# ==========================================
# STEP 16. Logistic Hyperparameter Grid
# ==========================================

param_grid = [

    # L2
    {
        "C": 0.01,
        "penalty": "l2",
        "solver": "liblinear"
    },

    {
        "C": 0.1,
        "penalty": "l2",
        "solver": "liblinear"
    },

    {
        "C": 1.0,
        "penalty": "l2",
        "solver": "liblinear"
    },

    {
        "C": 10.0,
        "penalty": "l2",
        "solver": "liblinear"
    },

    # L1
    {
        "C": 0.01,
        "penalty": "l1",
        "solver": "liblinear"
    },

    {
        "C": 0.1,
        "penalty": "l1",
        "solver": "liblinear"
    },

    {
        "C": 1.0,
        "penalty": "l1",
        "solver": "liblinear"
    },

    {
        "C": 10.0,
        "penalty": "l1",
        "solver": "liblinear"
    }
]


print(
    "튜닝 후보:",
    len(param_grid),
    "개"
)

In [ ]:
# ==========================================
# STEP 17. 3-Fold Hyperparameter Tuning
# ==========================================

tuning_results = []


for set_name, features in feature_sets.items():

    print("\n")
    print("=" * 80)
    print(
        "TUNING:",
        set_name
    )
    print("=" * 80)


    for params in param_grid:

        fold_pr_auc = []


        for fold_name, fold_info in folds.items():

            fold_train, fold_val = (
                get_fold_data(
                    segments,
                    fold_info
                )
            )


            X_train = fold_train[
                features
            ]

            y_train = fold_train[
                "is_fraud"
            ]

            X_val = fold_val[
                features
            ]

            y_val = fold_val[
                "is_fraud"
            ]


            model = make_logistic_pipeline(

                features=features,

                C=params["C"],

                penalty=params[
                    "penalty"
                ],

                solver=params[
                    "solver"
                ]
            )


            model.fit(
                X_train,
                y_train
            )


            val_prob = (
                model
                .predict_proba(
                    X_val
                )[:, 1]
            )


            pr_auc = (
                average_precision_score(
                    y_val,
                    val_prob
                )
            )


            fold_pr_auc.append(
                pr_auc
            )


        tuning_results.append({

            "Feature Set":
                set_name,

            "C":
                params["C"],

            "Penalty":
                params["penalty"],

            "Solver":
                params["solver"],

            "Fold 1 PR-AUC":
                fold_pr_auc[0],

            "Fold 2 PR-AUC":
                fold_pr_auc[1],

            "Fold 3 PR-AUC":
                fold_pr_auc[2],

            "Mean PR-AUC":
                np.mean(
                    fold_pr_auc
                ),

            "Std PR-AUC":
                np.std(
                    fold_pr_auc
                )
        })


tuning_results_df = pd.DataFrame(
    tuning_results
)


display(
    tuning_results_df.head()
)



TUNING: Set 1


TUNING: Set 2


TUNING: Set 3


TUNING: Set 4


TUNING: Set 5


In [ ]:
# ==========================================
# STEP 18. Set별 Best Parameter
# ==========================================

best_params_df = (

    tuning_results_df

    .sort_values(
        [
            "Feature Set",
            "Mean PR-AUC",
            "Std PR-AUC"
        ],

        ascending=[
            True,
            False,
            True
        ]
    )

    .groupby(
        "Feature Set",
        as_index=False
    )

    .first()
)


best_params_df = (
    best_params_df
    .sort_values(
        "Mean PR-AUC",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    best_params_df
)

In [ ]:
# ==========================================
# STEP 19. Best Parameter + Threshold 0.90
# ==========================================

final_cv_results = []


for _, row in best_params_df.iterrows():

    set_name = row[
        "Feature Set"
    ]

    features = feature_sets[
        set_name
    ]

    best_C = row["C"]

    best_penalty = row[
        "Penalty"
    ]

    best_solver = row[
        "Solver"
    ]


    for fold_name, fold_info in folds.items():

        fold_train, fold_val = (
            get_fold_data(
                segments,
                fold_info
            )
        )


        X_train = fold_train[
            features
        ]

        y_train = fold_train[
            "is_fraud"
        ]

        X_val = fold_val[
            features
        ]

        y_val = fold_val[
            "is_fraud"
        ]


        model = make_logistic_pipeline(

            features=features,

            C=best_C,

            penalty=best_penalty,

            solver=best_solver
        )


        model.fit(
            X_train,
            y_train
        )


        val_prob = (
            model
            .predict_proba(
                X_val
            )[:, 1]
        )


        metrics = evaluate_model(
            y_val,
            val_prob,
            threshold=0.90
        )


        final_cv_results.append({

            "Feature Set":
                set_name,

            "Fold":
                fold_name,

            "C":
                best_C,

            "Penalty":
                best_penalty,

            "Threshold":
                0.90,

            **metrics
        })


final_cv_results_df = pd.DataFrame(
    final_cv_results
)


display(
    final_cv_results_df
)

In [ ]:
# ==========================================
# STEP 20. 최종 성능 요약
# ==========================================

final_summary_df = (

    final_cv_results_df

    .groupby(
        "Feature Set"
    )

    .agg(

        Mean_PR_AUC=(
            "PR-AUC",
            "mean"
        ),

        Std_PR_AUC=(
            "PR-AUC",
            "std"
        ),

        Mean_Precision=(
            "Precision",
            "mean"
        ),

        Mean_Recall=(
            "Recall",
            "mean"
        ),

        Mean_F1=(
            "F1",
            "mean"
        ),

        Total_FP=(
            "FP",
            "sum"
        ),

        Total_FN=(
            "FN",
            "sum"
        )
    )

    .reset_index()
)


final_summary_df = (

    final_summary_df

    .sort_values(
        [
            "Mean_PR_AUC",
            "Std_PR_AUC",
            "Mean_F1"
        ],

        ascending=[
            False,
            True,
            False
        ]
    )

    .reset_index(drop=True)
)


final_summary_df.insert(
    0,
    "Rank",
    range(
        1,
        len(final_summary_df) + 1
    )
)


display(
    final_summary_df
)